# LKS AI Kabupaten Malang 2025 — Modul A, B, C, D

Demo lengkap: persiapan data & EDA, klasifikasi Decision Tree manual, evaluasi, dan prediksi data baru.

Logika model berada di `model.py`. Atur `INTERAKTIF = True` di sel Modul D untuk mencoba input pasien sendiri secara manual.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from model import (DEFAULT_DATASET, FEATURE_NAMES, TARGET,
                   feature_importances, predict_one, train)
import sys
sys.path.insert(0, '.')


## Modul A — Persiapan Data & EDA


In [ ]:
df = pd.read_csv(DEFAULT_DATASET, encoding='utf-8-sig').dropna()
print('Bentuk dataset:', df.shape)
print('\nNilai kosong per kolom:')
print(df.isnull().sum().to_string())
print('\nStatistik deskriptif:')
print(df.describe().to_string())
print('\nDistribusi target:')
print(df[TARGET].value_counts().to_string())


In [ ]:
numeric = df.select_dtypes(include=[np.number]).columns.tolist()
if TARGET in numeric:
    numeric.remove(TARGET)
df[numeric].hist(bins=15, figsize=(15, 5))
plt.suptitle('Distribusi Fitur Numerik')
plt.tight_layout(); plt.show()
plt.figure(figsize=(12, 10))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Heatmap Korelasi Fitur')
plt.tight_layout(); plt.show()
plt.figure(figsize=(6, 4))
sns.countplot(x=df[TARGET])
plt.title('Distribusi Target'); plt.xlabel('Target'); plt.ylabel('Jumlah')
plt.tight_layout(); plt.show()


## Modul B — Klasifikasi (Decision Tree)


In [ ]:
tree, m = train(dataset=df)
print('Train size:', len(m['y_train']), ' Test size:', len(m['y_test']))
from model import print_tree
print('\nStruktur pohon keputusan:')
print_tree(tree)


## Modul C — Evaluasi Model


In [ ]:
print(f"\nAkurasi  : {m['acc']:.4f}")
print(f"Presisi  : {m['prec']:.4f}")
print(f"Recall   : {m['rec']:.4f}")
print(f"F1-Score : {m['f1']:.4f}")
print('Matriks konfusi (baris=aktual, kolom=prediksi):')
print(m['cm'])
plt.figure(figsize=(8, 6))
sns.heatmap(m['cm'], annot=True, fmt='d', cmap='Blues',
            xticklabels=m['classes'], yticklabels=m['classes'])
plt.title('Confusion Matrix'); plt.xlabel('Prediksi'); plt.ylabel('Aktual')
plt.tight_layout(); plt.show()
imp = feature_importances(tree)
items = list(imp.items())
plt.figure(figsize=(10, 4))
sns.barplot(x=[v for _, v in items], y=[k for k, _ in items], color='#4c72b0')
plt.title('Feature Importance (information gain)')
plt.xlabel('Total gain tertimbang jumlah sampel')
plt.tight_layout(); plt.show()


## Modul D — Uji Coba Data Baru (Prediksi)

Tiga contoh pasien di bawah ini menampilkan hasil prediksi. Untuk input interaktif, ubah `INTERAKTIF = True` lalu jalankan ulang sel.


In [ ]:
contoh = [
    {'age':63,'sex':1,'cp':3,'trestbps':145,'chol':233,'fbs':1,'restecg':0,
     'thalach':150,'exang':0,'oldpeak':2.3,'slope':0,'ca':0,'thal':1},
    {'age':57,'sex':0,'cp':0,'trestbps':140,'chol':241,'fbs':0,'restecg':1,
     'thalach':123,'exang':1,'oldpeak':0.2,'slope':1,'ca':0,'thal':3},
    {'age':53,'sex':1,'cp':0,'trestbps':130,'chol':197,'fbs':1,'restecg':0,
     'thalach':152,'exang':0,'oldpeak':1.2,'slope':0,'ca':0,'thal':3},
]
for i, row in enumerate(contoh, 1):
    print(f"Contoh {i}: prediksi {predict_one(tree, row)}")

INTERAKTIF = False  # ubah ke True untuk coba input manual di notebook
if INTERAKTIF:
    print('\nMasukkan nilai untuk tiap fitur (kosongkan lalu ENTER untuk keluar):')
    while True:
        raw = input('\nBaris baru? (y/n): ').strip().lower()
        if raw not in ('y', 'yes'):
            break
        row = {}
        for name in FEATURE_NAMES:
            val = input(f"  {name}: ").strip()
            if not val:
                break
            row[name] = float(val)
        if len(row) == len(FEATURE_NAMES):
            print('Prediksi:', predict_one(tree, row))
        else:
            print('Input tidak lengkap.')
